<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/13SOLID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_solid.py

from __future__ import annotations

from dataclasses import dataclass, field
from enum import StrEnum, verify, UNIQUE
from pathlib import Path
from typing import Final, Protocol, runtime_checkable

from modul_filtry import (
    BaseFilter,
    FiltrMomentum,
    SpolkaDoFiltrow,
    StatusFiltra,
    WynikFiltra,
)

from modul_skaner import (
    JsonScannerRepository,
)


FOLDER_DANYCH: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


@verify(UNIQUE)
class PoziomLogowania(StrEnum):
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"


@verify(UNIQUE)
class FormatEksportu(StrEnum):
    TEXT = "text"


@runtime_checkable
class RepositoryProtocol(Protocol):

    def pobierz_spolke(
        self,
        ticker: str,
    ) -> SpolkaDoFiltrow:
        ...


@runtime_checkable
class FilterProtocol(Protocol):

    def ocen(
        self,
        spolka: SpolkaDoFiltrow,
    ) -> WynikFiltra:
        ...


@runtime_checkable
class LoggerProtocol(Protocol):

    def log(
        self,
        komunikat: str,
        poziom: PoziomLogowania = PoziomLogowania.INFO,
    ) -> None:
        ...


@runtime_checkable
class ExportProtocol(Protocol):

    def eksportuj(
        self,
        wynik: WynikAnalizySOLID,
    ) -> str:
        ...


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class WynikAnalizySOLID:

    ticker: str

    przeszedl: bool

    wyniki_filtrow: tuple[
        WynikFiltra,
        ...
    ] = field(
        default_factory=tuple
    )

    liczba_filtrow: int = field(
        init=False
    )

    def __post_init__(self) -> None:

        object.__setattr__(
            self,
            "liczba_filtrow",
            len(self.wyniki_filtrow),
        )


class ConsoleLogger:

    def log(
        self,
        komunikat: str,
        poziom: PoziomLogowania = PoziomLogowania.INFO,
    ) -> None:

        print(
            f"[{poziom.value.upper()}]",
            komunikat,
        )


class NullLogger:

    def log(
        self,
        komunikat: str,
        poziom: PoziomLogowania = PoziomLogowania.INFO,
    ) -> None:

        return None


class LoggingMixin:

    logger: LoggerProtocol

    def log_info(
        self,
        komunikat: str,
    ) -> None:

        self.logger.log(
            komunikat,
            PoziomLogowania.INFO,
        )

    def log_warning(
        self,
        komunikat: str,
    ) -> None:

        self.logger.log(
            komunikat,
            PoziomLogowania.WARNING,
        )

    def log_error(
        self,
        komunikat: str,
    ) -> None:

        self.logger.log(
            komunikat,
            PoziomLogowania.ERROR,
        )


class ValidationMixin:

    def waliduj_ticker(
        self,
        ticker: str,
    ) -> str:

        poprawiony: str = (
            ticker
            .strip()
            .upper()
        )

        if not poprawiony:

            raise ValueError(
                "ticker nie moze byc pusty"
            )

        if not poprawiony.replace(
            ".",
            "",
        ).replace(
            "-",
            "",
        ).isalnum():

            raise ValueError(
                f"niepoprawny ticker: {ticker}"
            )

        return poprawiony


class ExportMixin:

    def eksportuj(
        self,
        wynik: WynikAnalizySOLID,
    ) -> str:

        status: str = (
            StatusFiltra.PASSED.value
            if wynik.przeszedl
            else StatusFiltra.FAILED.value
        )

        return (
            f"ticker={wynik.ticker};"
            f"status={status};"
            f"liczba_filtrow="
            f"{wynik.liczba_filtrow}"
        )


class BaseLifecycle:

    def uruchomienie(
        self,
    ) -> list[str]:

        return [
            "BaseLifecycle"
        ]


class LoggingLifecycleMixin:

    def uruchomienie(
        self,
    ) -> list[str]:

        kroki: list[str] = (
            super().uruchomienie()
        )

        kroki.append(
            "LoggingLifecycleMixin"
        )

        return kroki


class ValidationLifecycleMixin:

    def uruchomienie(
        self,
    ) -> list[str]:

        kroki: list[str] = (
            super().uruchomienie()
        )

        kroki.append(
            "ValidationLifecycleMixin"
        )

        return kroki


class LifecycleDemo(
    LoggingLifecycleMixin,
    ValidationLifecycleMixin,
    BaseLifecycle,
):
    pass


class ZaawansowanyFiltrMomentum(
    FiltrMomentum,
):

    def __init__(
        self,
        minimalne_momentum: float = 0.0,
        minimalny_wolumen: float = 100_000.0,
    ) -> None:

        super().__init__(
            minimalne_momentum=(
                minimalne_momentum
            )
        )

        self.minimalny_wolumen = (
            minimalny_wolumen
        )

    def ocen(
        self,
        spolka: SpolkaDoFiltrow,
    ) -> WynikFiltra:

        wynik_podstawowy: WynikFiltra = (
            super().ocen(
                spolka
            )
        )

        momentum_ok: bool = (
            wynik_podstawowy.status
            == StatusFiltra.PASSED
        )

        wolumen_ok: bool = (
            spolka.sredni_wolumen
            >= self.minimalny_wolumen
        )

        if (
            momentum_ok
            and wolumen_ok
        ):

            return wynik_podstawowy

        return WynikFiltra(
            typ_filtra=(
                wynik_podstawowy
                .typ_filtra
            ),
            status=(
                StatusFiltra.FAILED
            ),
            wartosc=(
                wynik_podstawowy
                .wartosc
            ),
            prog=(
                wynik_podstawowy
                .prog
            ),
            opis=(
                "rozszerzony filtr momentum: "
                "momentum oraz minimalny wolumen"
            ),
        )


class SolidSkaner(
    LoggingMixin,
    ValidationMixin,
    ExportMixin,
):

    def __init__(
        self,
        *,
        repository: RepositoryProtocol,
        filtry: list[FilterProtocol],
        logger: LoggerProtocol,
    ) -> None:

        if not filtry:

            raise ValueError(
                "lista filtrow nie moze byc pusta"
            )

        self.repository = repository
        self.filtry = list(
            filtry
        )
        self.logger = logger

    def pobierz_spolke(
        self,
        ticker: str,
    ) -> SpolkaDoFiltrow:

        ticker = self.waliduj_ticker(
            ticker
        )

        self.log_info(
            f"pobieranie danych dla {ticker}"
        )

        return (
            self.repository
            .pobierz_spolke(
                ticker
            )
        )

    def ocen_spolke(
        self,
        spolka: SpolkaDoFiltrow,
    ) -> tuple[
        WynikFiltra,
        ...
    ]:

        wyniki: list[
            WynikFiltra
        ] = []

        for filtr in self.filtry:

            wynik: WynikFiltra = (
                filtr.ocen(
                    spolka
                )
            )

            wyniki.append(
                wynik
            )

            self.log_info(
                f"{spolka.ticker}: "
                f"{wynik.typ_filtra.value} "
                f"-> "
                f"{wynik.status.value}"
            )

        return tuple(
            wyniki
        )

    def podejmij_decyzje(
        self,
        wyniki: tuple[
            WynikFiltra,
            ...
        ],
    ) -> bool:

        if not wyniki:

            raise ValueError(
                "brak wynikow filtrow"
            )

        return all(
            wynik.status
            == StatusFiltra.PASSED

            for wynik
            in wyniki
        )

    def skanuj(
        self,
        ticker: str,
    ) -> WynikAnalizySOLID:

        spolka: SpolkaDoFiltrow = (
            self.pobierz_spolke(
                ticker
            )
        )

        wyniki: tuple[
            WynikFiltra,
            ...
        ] = self.ocen_spolke(
            spolka
        )

        decyzja: bool = (
            self.podejmij_decyzje(
                wyniki
            )
        )

        if decyzja:

            self.log_info(
                f"{spolka.ticker} "
                "przeszla wszystkie filtry"
            )

        else:

            self.log_warning(
                f"{spolka.ticker} "
                "nie przeszla wszystkich filtrow"
            )

        return WynikAnalizySOLID(
            ticker=spolka.ticker,
            przeszedl=decyzja,
            wyniki_filtrow=wyniki,
        )


class SolidScannerFactory:

    @classmethod
    def create(
        cls,
        *,
        repository: RepositoryProtocol,
        filtry: list[FilterProtocol],
        logger: LoggerProtocol,
    ) -> SolidSkaner:

        return SolidSkaner(
            repository=repository,
            filtry=filtry,
            logger=logger,
        )


def pokaz_mro() -> None:

    print(
        "\nMRO LifecycleDemo:"
    )

    for klasa in (
        LifecycleDemo.__mro__
    ):

        print(
            "-",
            klasa.__name__,
        )

    demo = LifecycleDemo()

    print(
        "\nKOLEJNOSC super():"
    )

    for krok in (
        demo.uruchomienie()
    ):

        print(
            "-",
            krok,
        )


def pokaz_interfejsy(
    repository: RepositoryProtocol,
    filtr: FilterProtocol,
    logger: LoggerProtocol,
) -> None:

    print(
        "\nISP / PROTOCOL:"
    )

    print(
        "repository zgodne:",
        isinstance(
            repository,
            RepositoryProtocol,
        ),
    )

    print(
        "filtr zgodny:",
        isinstance(
            filtr,
            FilterProtocol,
        ),
    )

    print(
        "logger zgodny:",
        isinstance(
            logger,
            LoggerProtocol,
        ),
    )


def run() -> None:

    ticker: str = input(
        "podaj ticker: "
    ).strip().upper()

    repository: RepositoryProtocol = (
        JsonScannerRepository(
            folder=FOLDER_DANYCH
        )
    )

    filtr: BaseFilter = (
        ZaawansowanyFiltrMomentum(
            minimalne_momentum=0.0,
            minimalny_wolumen=100_000.0,
        )
    )

    logger: LoggerProtocol = (
        ConsoleLogger()
    )

    pokaz_interfejsy(
        repository,
        filtr,
        logger,
    )

    pokaz_mro()

    skaner: SolidSkaner = (
        SolidScannerFactory.create(
            repository=repository,
            filtry=[
                filtr
            ],
            logger=logger,
        )
    )

    wynik: WynikAnalizySOLID = (
        skaner.skanuj(
            ticker
        )
    )

    print(
        "\nWYNIK:"
    )

    print(
        wynik
    )

    print(
        "\nEKSPORT:"
    )

    print(
        skaner.eksportuj(
            wynik
        )
    )

    print(
        "\nMODUL SOLID "
        "DZIALA POPRAWNIE"
    )